# IAM Role Validation via Real SDK Interfaces (No Auto-Creation)

The SDK validates the role you pass (or your caller identity) and does not create
IAM roles as a side effect. By default, each interface resolves the role and
**validates** it. If the role is missing required permissions, you get a clear
`RoleValidationError` instead of a silently-created role.

To provision a least-privilege role on purpose, use the opt-in
`IamRoleResolver().create_execution_role(...)` utility (shown at the end).

⚠️ Runs against **real AWS**. Use a non-production account.

## Setup


In [ ]:
import sys

assert sys.version_info >= (3, 10), (
    f"This notebook needs Python 3.10+ (got {sys.version.split()[0]})."
)

for p in (
    "../sagemaker-core/src",
    "../sagemaker-train/src",
    "../sagemaker-serve/src",
    "../sagemaker-mlops/src",
):
    if p not in sys.path:
        sys.path.insert(0, p)

import logging
import uuid
import boto3
from botocore.exceptions import ClientError

logging.getLogger().setLevel(logging.WARNING)

iam = boto3.client("iam")
sts = boto3.client("sts")
REGION = boto3.Session().region_name or "us-west-2"
ACCOUNT = sts.get_caller_identity()["Account"]

from sagemaker.core.helper import (
    IamRoleResolver,
    resolve_and_validate_role,
    RoleValidationError,
)

# A role name that does not exist, so validation must fail (never auto-create).
NONEXISTENT_ROLE = f"sdk-does-not-exist-{uuid.uuid4().hex[:12]}"

def autorole_exists(name):
    try:
        iam.get_role(RoleName=name); return True
    except ClientError as e:
        return e.response["Error"]["Code"] not in ("NoSuchEntity", "NoSuchEntityException")

print(f"Account: {ACCOUNT} | Region: {REGION}")
print(f"Identity: {sts.get_caller_identity()['Arn']}")
print("\n→ Confirm this is a NON-PRODUCTION account before continuing.")

## 1. `ModelTrainer` — a missing role raises `RoleValidationError`, creates nothing


In [ ]:
from sagemaker.train.model_trainer import ModelTrainer

try:
    ModelTrainer(
        training_image=f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/img:latest",
        role=NONEXISTENT_ROLE,
    ).train()
    raise AssertionError("expected a validation error")
except (RoleValidationError, ValueError) as e:
    print("Got expected error:\n", e)

assert not autorole_exists("SageMaker-AutoRole-Training"), "no role should be auto-created"
print("\nOK: no IAM role was created as a side effect.")

## 2. `SFTTrainer` — same validate-only behavior on `.train()`


In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType

try:
    SFTTrainer(
        model="nova-textgeneration-lite-v2",
        training_type=TrainingType.LORA,
        training_dataset="s3://example-bucket/train.jsonl",
        role=NONEXISTENT_ROLE,
    ).train(wait=False)
    raise AssertionError("expected a validation error")
except (RoleValidationError, ValueError) as e:
    print("Got expected error:\n", e)

assert not autorole_exists("SageMaker-AutoRole-Training")
print("\nOK: no IAM role was created as a side effect.")

## 3. `ModelBuilder` — validate-only on `.build()`


In [ ]:
from sagemaker.serve.model_builder import ModelBuilder

try:
    ModelBuilder(
        model="meta-textgeneration-llama-3-8b",
        role_arn=NONEXISTENT_ROLE,
    ).build()
    raise AssertionError("expected a validation error")
except (RoleValidationError, ValueError) as e:
    print("Got expected error:\n", e)

assert not autorole_exists("SageMaker-AutoRole-Serving")
print("\nOK: no IAM role was created as a side effect.")

## 4. Opt-in: create a least-privilege role on purpose

When you *want* a managed role, call the explicit utility. Preview the permissions
first, then create the role and pass its ARN into any interface.


In [ ]:
resolver = IamRoleResolver()

# Preview what the training role would be granted (read-only):
print("Required actions for 'training':")
for a in resolver.get_required_actions("training"):
    print("  ", a)

# Create the role (opt-in, scoped to a specific bucket):
role_arn = resolver.create_execution_role(
    role_type="training",
    s3_resource=f"sagemaker-{REGION}-{ACCOUNT}",
)
print("\nCreated:", role_arn)

# Now validation passes when you pass it explicitly:
validated = resolve_and_validate_role(provided_role=role_arn, role_type="training")
print("Validated:", validated)

## Cleanup


In [ ]:
# Remove the role created by the opt-in demo above.
IamRoleResolver().delete_execution_role(role_type="training")
print("Deleted SageMaker-AutoRole-Training (if it existed).")